# 04 — Naive and moving-average baselines

Last observation carried forward and MA 7/14/28. These are the models a planner should beat before celebrating Holt-Winters or ARIMA.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils.logging_config import setup_logging
from src.utils.helpers import load_config, load_model_config, resolve_path
setup_logging("INFO")
CONFIG = load_config()
print("Independent M5-schema project. DATA_DIR =", resolve_path(CONFIG["paths"]["data_dir"]))


In [ ]:
import pandas as pd
from src.forecasting.baseline import naive_forecast
from src.forecasting.moving_average import moving_average_forecast
from src.forecasting.evaluation import chronological_split, metrics_dict

fact = pd.read_parquet(resolve_path(CONFIG["paths"]["processed_dir"]) / "fact_daily_sales.parquet")
item, store = fact.groupby(["item_id", "store_id"])["revenue"].sum().idxmax()
series = fact[(fact.item_id == item) & (fact.store_id == store)].sort_values("date")
train, valid = chronological_split(series, "date", 28)
y = valid["sales_units"]
print(item, store)
print("Naive", metrics_dict(y, naive_forecast(train["sales_units"], 28)))
print("MA28", metrics_dict(y, moving_average_forecast(train["sales_units"], 28, 28)))
